### [Monte Carlo Risk Simulator (Simple Scenario Engine)](https://pub.towardsai.net/build-a-monte-carlo-risk-simulator-simple-scenario-engine-python-solution-f4ee48d4bcd7)

> Simple scenario engine for VaR, CVaR, and stress testing

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True)
pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Deterministic inputs
positions = pd.DataFrame({
    "asset": ["SPX", "TLT", "GLD"],
    "weight": [0.55, 0.30, 0.15],
    "mu_annual": [0.07, 0.03, 0.04],
    "vol_annual": [0.18, 0.12, 0.15],
})

corr = np.array([
    [1.00, -0.25, 0.10],
    [-0.25, 1.00, 0.05],
    [0.10, 0.05, 1.00],
])

# Pretty markdown tables (so your output looks the same everywhere)
pos_show = positions.copy()
pos_show["weight"] = pos_show["weight"].map(lambda x: f"{x:.2f}")
pos_show["mu_annual"] = pos_show["mu_annual"].map(lambda x: f"{x:.2%}")
pos_show["vol_annual"] = pos_show["vol_annual"].map(lambda x: f"{x:.2%}")

corr_df = pd.DataFrame(corr, index=positions["asset"], columns=positions["asset"])
corr_show = corr_df.copy()
corr_show.insert(0, "asset", corr_show.index)
for c in corr_show.columns[1:]:
    corr_show[c] = corr_show[c].map(lambda x: f"{x:.2f}")

def to_md(df):
    cols = list(df.columns)
    lines = []
    lines.append("| " + " | ".join(cols) + " |")
    lines.append("| " + " | ".join(["---"] * len(cols)) + " |")
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(map(str, r.values)) + " |")
    return "\n".join(lines)

print("Positions")
display(pos_show)
print("\nCorrelation")
display(corr_show)

Positions


,asset,weight,mu_annual,vol_annual
0,SPX,0.55,7.00%,18.00%
1,TLT,0.30,3.00%,12.00%
2,GLD,0.15,4.00%,15.00%



Correlation


asset,asset,SPX,TLT,GLD
asset,,,,
SPX,SPX,1.00,-0.25,0.10
TLT,TLT,-0.25,1.00,0.05
GLD,GLD,0.10,0.05,1.00


In [3]:
# --- Inputs (same every run) ---
positions = pd.DataFrame({
    "asset": ["SPX", "TLT", "GLD"],
    "weight": [0.55, 0.30, 0.15],
    "mu_annual": [0.07, 0.03, 0.04],
    "vol_annual": [0.18, 0.12, 0.15],
})

corr = np.array([
    [1.00, -0.25, 0.10],
    [-0.25, 1.00, 0.05],
    [0.10, 0.05, 1.00],
])

V0 = 10_000_000
H = 20           # business days
N = 20_000       # simulations
SEED = 7

w = positions["weight"].to_numpy()
mu = positions["mu_annual"].to_numpy()
vol = positions["vol_annual"].to_numpy()

# --- Scenario knobs ---
def push_corr_to_one(c, alpha):
    ones = np.ones_like(c)
    out = (1 - alpha) * c + alpha * ones
    np.fill_diagonal(out, 1.0)
    return out

scenarios = {
    "Base": {
        "mu_annual": mu,
        "vol_annual": vol,
        "corr": corr
    },
    "Stress": {
        "mu_annual": np.array([-0.06, 0.015, 0.02]),   # drift shock
        "vol_annual": vol * 1.6,                      # volatility shock
        "corr": push_corr_to_one(corr, alpha=0.35)     # correlation shock
    }
}

# --- Monte Carlo engine (vectorized) ---
def annual_to_dt(mu_annual, vol_annual, steps_per_year=252):
    dt = 1 / steps_per_year
    mu_log = np.log1p(mu_annual)
    drift = (mu_log - 0.5 * (vol_annual ** 2)) * dt
    vol_dt = vol_annual * np.sqrt(dt)
    return drift, vol_dt

def simulate_terminal_values(V0, w, mu_annual, vol_annual, corr, H, N, seed):
    rng = np.random.default_rng(seed)
    drift, vol_dt = annual_to_dt(mu_annual, vol_annual)

    L = np.linalg.cholesky(corr)
    Z = rng.standard_normal(size=(N, H, len(w)))
    eps = Z @ L.T

    log_r = drift + vol_dt * eps
    port_log_r = np.tensordot(log_r, w, axes=([2], [0]))      # (N, H)
    terminal = V0 * np.exp(port_log_r.sum(axis=1))
    return terminal

def risk_summary(V0, terminal):
    loss = V0 - terminal
    var95 = np.quantile(loss, 0.95)
    var99 = np.quantile(loss, 0.99)
    cvar95 = loss[loss >= var95].mean()
    cvar99 = loss[loss >= var99].mean()

    return {
        "expected_pnl": float((terminal - V0).mean()),
        "prob_loss": float((loss > 0).mean()),
        "VaR_95": float(var95),
        "CVaR_95": float(cvar95),
        "VaR_99": float(var99),
        "CVaR_99": float(cvar99),
    }

rows = []
for name, cfg in scenarios.items():
    terminal = simulate_terminal_values(V0, w, cfg["mu_annual"], cfg["vol_annual"], cfg["corr"], H, N, SEED)
    rows.append({"scenario": name, **risk_summary(V0, terminal)})

out = pd.DataFrame(rows)

def fmt_money(x):
    sign = "-" if x < 0 else ""
    return f"{sign}${abs(x):,.0f}"

out_show = out.copy()
out_show["expected_pnl"] = out_show["expected_pnl"].map(fmt_money)
out_show["prob_loss"] = out_show["prob_loss"].map(lambda x: f"{x:.2%}")
for c in ["VaR_95", "CVaR_95", "VaR_99", "CVaR_99"]:
    out_show[c] = out_show[c].map(fmt_money)

print(f"Monte Carlo risk summary (V0=${V0:,}, horizon={H}d, sims={N:,})")
display(out_show)

Monte Carlo risk summary (V0=$10,000,000, horizon=20d, sims=20,000)


,scenario,expected_pnl,prob_loss,VaR_95,CVaR_95,VaR_99,CVaR_99
0,Base,"$33,997",45.64%,"$433,660","$545,497","$615,663","$705,240"
1,Stress,"-$33,720",53.45%,"$925,903","$1,132,640","$1,260,542","$1,424,709"


In [4]:
import sys
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.ticker import FuncFormatter

# Same inputs and simulator (base scenario)
positions = pd.DataFrame({
    "asset": ["SPX", "TLT", "GLD"],
    "weight": [0.55, 0.30, 0.15],
    "mu_annual": [0.07, 0.03, 0.04],
    "vol_annual": [0.18, 0.12, 0.15],
})

corr = np.array([    [1.00, -0.25, 0.10],
    [-0.25, 1.00, 0.05],
    [0.10, 0.05, 1.00],
])

V0 = 10_000_000
H = 20
N = 20_000
SEED = 7

w = positions["weight"].to_numpy()
mu = positions["mu_annual"].to_numpy()
vol = positions["vol_annual"].to_numpy()

def annual_to_dt(mu_annual, vol_annual, steps_per_year=252):
    dt = 1 / steps_per_year
    mu_log = np.log1p(mu_annual)
    drift = (mu_log - 0.5 * (vol_annual ** 2)) * dt
    vol_dt = vol_annual * np.sqrt(dt)
    return drift, vol_dt

def simulate_terminal_values(V0, w, mu_annual, vol_annual, corr, H, N, seed):
    rng = np.random.default_rng(seed)
    drift, vol_dt = annual_to_dt(mu_annual, vol_annual)

    L = np.linalg.cholesky(corr)
    Z = rng.standard_normal(size=(N, H, len(w)))
    eps = Z @ L.T

    log_r = drift + vol_dt * eps
    port_log_r = np.tensordot(log_r, w, axes=([2], [0]))
    terminal = V0 * np.exp(port_log_r.sum(axis=1))
    return terminal

terminal = simulate_terminal_values(V0, w, mu, vol, corr, H, N, SEED)
loss = V0 - terminal

var95 = float(np.quantile(loss, 0.95))
cvar95 = float(loss[loss >= var95].mean())

# --- Animation settings (professional + deterministic) ---
gif_path = "mc_loss_convergence_1200x675.gif"
png_path = "mc_loss_convergence_cover_1200x675.png"

# Fixed bins and x-limits for consistent frames
bins = np.linspace(loss.min(), loss.max(), 64)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
bin_width = bins[1] - bins[0]

sample_sizes = np.linspace(400, len(loss), 36, dtype=int)

# 1200x675 exactly: 8in * 150dpi = 1200px, 4.5in * 150dpi = 675px
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)

# Formatters
money = FuncFormatter(lambda x, pos: f"${x:,.0f}")

# Pre-create bars (avoid clearing each frame -> less flicker, more pro)
counts0, _ = np.histogram(loss[:sample_sizes[0]], bins=bins, density=True)
bars = ax.bar(
    bin_centers, counts0, width=bin_width * 0.98,
    align="center", linewidth=0.6, alpha=0.95
)

# VaR line (fixed)
var_line = ax.axvline(var95, linestyle="--", linewidth=2.0)
var_label = ax.text(
    var95, 0, "  VaR 95%", rotation=90,
    va="bottom", ha="left", fontsize=9
)

# Layout and styling (professional but clean)
ax.set_title("Monte Carlo 20-Day Loss Distribution (Convergence)", fontsize=13, pad=10)
ax.set_xlabel("Loss over 20 trading days", fontsize=10)
ax.set_ylabel("Density", fontsize=10)
ax.xaxis.set_major_formatter(money)
ax.grid(True, alpha=0.18, linewidth=0.8)

# Remove cluttered spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Stable axis limits (so it doesn't jump around)
xpad = 0.03 * (loss.max() - loss.min())
ax.set_xlim(loss.min() - xpad, loss.max() + xpad)

# Use a robust y-limit based on the final histogram (stable scaling)
final_counts, _ = np.histogram(loss, bins=bins, density=True)
ax.set_ylim(0, final_counts.max() * 1.15)

# Info box (updates every frame)
info = ax.text(
    0.02, 0.96, "", transform=ax.transAxes,
    va="top", ha="left", fontsize=9,
    bbox=dict(boxstyle="round,pad=0.35", alpha=0.12, linewidth=0.8)
)

# Place VaR label near the top (after y-limit is set)
var_label.set_position((var95, ax.get_ylim()[1] * 0.06))

# Precompute a few stable numbers for the box
prob_loss = float((loss > 0).mean())

def update(i):
    n = int(sample_sizes[i])
    counts, _ = np.histogram(loss[:n], bins=bins, density=True)

    # Update bar heights only
    for rect, h in zip(bars, counts):
        rect.set_height(h)

    # Update info box text
    info.set_text(
        f"Paths: {n:,}\n"
        f"VaR 95%: ${var95:,.0f}\n"
        f"CVaR 95%: ${cvar95:,.0f}\n"
        f"P(Loss>0): {prob_loss:.2%}"
    )

    return (*bars, info, var_line, var_label)

# Save a clean cover image (use the last frame)
update(len(sample_sizes) - 1)
fig.tight_layout()
if "google.colab" not in sys.modules:
  fig.savefig(png_path, bbox_inches="tight")

# Build and save GIF
anim = FuncAnimation(fig, update, frames=len(sample_sizes), interval=110, blit=False, repeat=True)
if "google.colab" not in sys.modules:
  anim.save(gif_path, writer=PillowWriter(fps=9))
else:
  from matplotlib import rc
  rc('animation', html='jshtml')
  display(anim)

plt.close(fig)

In [5]:
# Same inputs
positions = pd.DataFrame({
    "asset": ["SPX", "TLT", "GLD"],
    "weight": [0.55, 0.30, 0.15],
    "mu_annual": [0.07, 0.03, 0.04],
    "vol_annual": [0.18, 0.12, 0.15],
})

corr = np.array([
    [1.00, -0.25, 0.10],
    [-0.25, 1.00, 0.05],
    [0.10, 0.05, 1.00],
])

V0 = 10_000_000
H = 20
N = 20_000
SEED = 7

w = positions["weight"].to_numpy()
mu = positions["mu_annual"].to_numpy()
sigma = positions["vol_annual"].to_numpy()

def annual_to_dt(mu_annual, vol_annual, steps_per_year=252):
    dt = 1 / steps_per_year
    mu_log = np.log1p(mu_annual)
    drift = (mu_log - 0.5 * (vol_annual ** 2)) * dt
    vol_dt = vol_annual * np.sqrt(dt)
    return drift, vol_dt

def simulate_terminal_values(V0, w, mu_annual, vol_annual, corr, H, N, seed):
    rng = np.random.default_rng(seed)
    drift, vol_dt = annual_to_dt(mu_annual, vol_annual)

    L = np.linalg.cholesky(corr)
    Z = rng.standard_normal(size=(N, H, len(w)))
    eps = Z @ L.T

    log_r = drift + vol_dt * eps
    port_log_r = np.tensordot(log_r, w, axes=([2], [0]))
    terminal = V0 * np.exp(port_log_r.sum(axis=1))
    return terminal

terminal = simulate_terminal_values(V0, w, mu, sigma, corr, H, N, SEED)
log_sum = np.log(terminal / V0)

# Theoretical moments for sum of portfolio log-returns
dt = 1 / 252
drift, _ = annual_to_dt(mu, sigma)

cov_daily = np.outer(sigma, sigma) * corr * dt
mean_theory = H * float(w @ drift)
var_theory = H * float(w @ cov_daily @ w)

mean_sim = float(log_sum.mean())
var_sim = float(log_sum.var(ddof=0))

check = pd.DataFrame([
    {"metric": "mean(log return sum)", "theory": mean_theory, "simulation": mean_sim},
    {"metric": "var(log return sum)", "theory": var_theory, "simulation": var_sim},
])

check["theory"] = check["theory"].map(lambda x: f"{x:.6f}")
check["simulation"] = check["simulation"].map(lambda x: f"{x:.6f}")


print("Quick verification: theory vs simulation (base scenario)")
display(check)

Quick verification: theory vs simulation (base scenario)


,metric,theory,simulation
0,mean(log return sum),0.003112,0.002980
1,var(log return sum),0.000821,0.000828
